# Flatten `tags_combine_final_source` → Long Table

把源表的嵌套结构展开成一张「长表」（方案 B：一行 = 一个主体的一个 tag）。

| 项 | 值 |
|---|---|
| 源表 | `vehicel_test_source.Oringinal.tags_combine_final_source` |
| 目标表 | `vehicel_test_source.Oringinal.tags_combine_final_flattened` |
| 写入模式 | `CREATE OR REPLACE TABLE`（每次重跑全量覆盖） |
| 行粒度 | 一行 = 一个 `unify_id` × 一个 `tag_group` × 一个 `tag_id` |

目标表 schema：

| 列 | 类型 | 说明 |
|---|---|---|
| `unify_id` | string | 来自 `unifyId.id` |
| `unify_type` | string | 来自 `unifyId.type` |
| `tag_group` | string | `stags` / `dtags` / `ntags` / `tags` |
| `tag_id` | int | 来自 `xxx.tagId` |
| `tag_value_str` | string | 仅 `stags` 有值，其它为 `null` |
| `tag_value_long` | bigint | 仅 `dtags` 有值，其它为 `null` |
| `tag_value_double` | double | 仅 `ntags` 有值，其它为 `null` |
| `pii` | smallint | 仅 `stags` 有值，其它为 `null` |

> 注意：`tags` 数组里只有 `tagId`，没有 value，所以三个 value 列都是 `null`。


In [ ]:
# ============================================================
# 1. 目标 catalog / schema / 表名
# ============================================================
CATALOG = "vehicel_test_source"
SCHEMA = "Oringinal"

SOURCE_TABLE = "tags_combine_final_source"
TARGET_TABLE = "tags_combine_final_flattened"

SOURCE_FQN = f"`{CATALOG}`.`{SCHEMA}`.`{SOURCE_TABLE}`"
TARGET_FQN = f"`{CATALOG}`.`{SCHEMA}`.`{TARGET_TABLE}`"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"USE SCHEMA  `{SCHEMA}`")

print("Source table :", SOURCE_FQN)
print("Target table :", TARGET_FQN)
display(spark.sql(f"DESCRIBE `{CATALOG}`.`{SCHEMA}`.`{SOURCE_TABLE}`"))


In [ ]:
# ============================================================
# 2. 展开 4 个 array 并 UNION ALL 成长表
# ============================================================
flatten_sql = f"""
CREATE OR REPLACE TABLE {TARGET_FQN} AS
SELECT
  unifyId.id   AS unify_id,
  unifyId.type AS unify_type,
  'stags' AS tag_group,
  s.tagId AS tag_id,
  s.value AS tag_value_str,
  CAST(NULL AS BIGINT)   AS tag_value_long,
  CAST(NULL AS DOUBLE)   AS tag_value_double,
  s.pii   AS pii
FROM {SOURCE_FQN}
LATERAL VIEW explode(tagRecord.stags) exploded AS s

UNION ALL

SELECT
  unifyId.id   AS unify_id,
  unifyId.type AS unify_type,
  'dtags' AS tag_group,
  d.tagId AS tag_id,
  CAST(NULL AS STRING)   AS tag_value_str,
  d.value AS tag_value_long,
  CAST(NULL AS DOUBLE)   AS tag_value_double,
  CAST(NULL AS SMALLINT) AS pii
FROM {SOURCE_FQN}
LATERAL VIEW explode(tagRecord.dtags) exploded AS d

UNION ALL

SELECT
  unifyId.id   AS unify_id,
  unifyId.type AS unify_type,
  'ntags' AS tag_group,
  n.tagId AS tag_id,
  CAST(NULL AS STRING)   AS tag_value_str,
  CAST(NULL AS BIGINT)   AS tag_value_long,
  n.value AS tag_value_double,
  CAST(NULL AS SMALLINT) AS pii
FROM {SOURCE_FQN}
LATERAL VIEW explode(tagRecord.ntags) exploded AS n

UNION ALL

SELECT
  unifyId.id   AS unify_id,
  unifyId.type AS unify_type,
  'tags' AS tag_group,
  t.tagId AS tag_id,
  CAST(NULL AS STRING)   AS tag_value_str,
  CAST(NULL AS BIGINT)   AS tag_value_long,
  CAST(NULL AS DOUBLE)   AS tag_value_double,
  CAST(NULL AS SMALLINT) AS pii
FROM {SOURCE_FQN}
LATERAL VIEW explode(tagRecord.tags) exploded AS t
"""

spark.sql(flatten_sql)
print(f"[OK] Created {TARGET_FQN}")


In [ ]:
# ============================================================
# 3. 验证：schema、行数、按 tag_group 分布、抽样
# ============================================================
print(f"--- {TARGET_FQN} schema ---")
spark.table(TARGET_FQN).printSchema()

total_rows = spark.table(TARGET_FQN).count()
print(f"\nTotal rows: {total_rows:,}")

print("\n--- rows per tag_group ---")
display(spark.sql(f"""
SELECT tag_group, COUNT(*) AS row_count
FROM {TARGET_FQN}
GROUP BY tag_group
ORDER BY tag_group
"""))

print("\n--- sample rows ---")
display(spark.sql(f"SELECT * FROM {TARGET_FQN} LIMIT 20"))

print("\n--- physical location ---")
display(spark.sql(f"DESCRIBE DETAIL {TARGET_FQN}").select("name", "format", "location", "numFiles", "sizeInBytes"))
